# 4 Per-Source Distance Computation (Alternative)
Compute haversine distances between all 4 original source coordinate columns
(not the grouped Webscraped approach used in the main pipeline). Useful for
comparison and debugging.

Input: data/1_derived/5_geocode_truck_stops/2_cross_source_distances.csv

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
from math import radians, cos, sin, asin, sqrt


def find_project_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / 'data').exists() and (candidate / 'source').exists() and (candidate / 'temp').exists():
            return candidate
    return start


PROJECT_ROOT = find_project_root(Path.cwd())
IN_FILE = PROJECT_ROOT / 'data' / '1_derived' / '5_geocode_truck_stops' / '2_cross_source_distances.csv'

df = pd.read_csv(IN_FILE, low_memory=False)
print(f'Loaded {len(df):,} rows')

columns_info = [
    ('Webscraped_Phone_LD_Latitude', 'Webscraped_Phone_LD_Longitude', 'Truckstopsandservices (Phone)'),
    ('Webscraped_PlacedMatched_LD_Latitude', 'Webscraped_PlacedMatched_LD_Longitude', 'Truckstopsandservices (PlaceMatched)'),
    ('Yelp_Latitude', 'Yelp_Longitude', 'Yelp (Phone)'),
    ('YellowPages_JSONLD_LAT_1', 'YellowPages_JSONLD_LNG_1', 'YellowPages (Phone)')
]

def parse_coords(lat_str, lon_str):
    exclude = {'nan', '', 'None', 'removed'}
    lats = [float(x) for x in str(lat_str).split(';') if x.strip() not in exclude]
    lons = [float(x) for x in str(lon_str).split(';') if x.strip() not in exclude]
    return list(zip(lats, lons))

def haversine(lat1, lon1, lat2, lon2):
    R = 3958.8
    dlat = radians(lat2 - lat1)
    dlon = radians(lon2 - lon1)
    a = sin(dlat / 2) ** 2 + cos(radians(lat1)) * cos(radians(lat2)) * sin(dlon / 2) ** 2
    return R * 2 * asin(sqrt(a))

max_distances = []
max_distance_sources = []
min_distances = []
min_distance_sources = []

for idx, row in df.iterrows():
    source_coords = {}
    for lat_col, lon_col, label in columns_info:
        if lat_col in row and lon_col in row:
            coords = parse_coords(row[lat_col], row[lon_col])
            if coords:
                source_coords[label] = coords
    max_dist = 0
    min_dist = float('inf')
    max_pair = ''
    min_pair = ''
    names = list(source_coords.keys())
    for i in range(len(names)):
        for j in range(i + 1, len(names)):
            for lat1, lon1 in source_coords[names[i]]:
                for lat2, lon2 in source_coords[names[j]]:
                    d = haversine(lat1, lon1, lat2, lon2)
                    if d > max_dist:
                        max_dist = d
                        max_pair = f'{names[i]} vs {names[j]}'
                    if d < min_dist:
                        min_dist = d
                        min_pair = f'{names[i]} vs {names[j]}'
    max_distances.append(max_dist)
    max_distance_sources.append(max_pair)
    min_distances.append(min_dist if min_dist != float('inf') else 0)
    min_distance_sources.append(min_pair)

df['alt_max_distance_miles'] = max_distances
df['alt_max_distance_sources'] = max_distance_sources
df['alt_min_distance_miles'] = min_distances
df['alt_min_distance_sources'] = min_distance_sources

print('Per-source distance computation complete.')
print(df[['alt_min_distance_miles', 'alt_min_distance_sources',
          'alt_max_distance_miles', 'alt_max_distance_sources']].describe())